# BPE v10 Oluşturma Notebook'u

Bu notebook şunları yapar:
1. `new_custom_bpe_tokenizer_yusuf.json` dosyasından vocab çıkarır
2. Test veri setlerinde token frekanslarını hesaplar
3. Frekans analizine dayalı olarak en sık kullanılan tokenleri seçer
4. `bpe_v10.json` dosyasını oluşturur (ID aralığı: 22869-32767)

## Metodoloji:
- Mevcut tokenizer'dan vocab'u çıkar
- Test metinlerinde tokenizasyon yap
- Token frekanslarını hesapla
- Frekans eşiği belirle (minimum kullanım sayısı)
- En sık kullanılan tokenleri yeni BPE'ye ekle


In [ ]:
# Gerekli kütüphaneleri import et
import json
import os
import re
from collections import Counter, defaultdict
from datasets import load_dataset
from tokenizers import Tokenizer
import pandas as pd
from tqdm import tqdm
import numpy as np


## 1. Mevcut Tokenizer'ı Yükle


In [ ]:
# new_custom_bpe_tokenizer_yusuf.json dosyasını yükle
print("🔄 Tokenizer dosyası yükleniyor...")
with open('new_custom_bpe_tokenizer_yusuf.json', 'r', encoding='utf-8') as f:
    tokenizer_data = json.load(f)

# Vocab kısmını çıkar
existing_vocab = tokenizer_data['model']['vocab']
print(f"📊 Mevcut vocab boyutu: {len(existing_vocab):,}")

# Tokenizer objesini oluştur
tokenizer = Tokenizer.from_file('new_custom_bpe_tokenizer_yusuf.json')
print("✅ Tokenizer başarıyla yüklendi")

# Vocab örnekleri göster
print("\n📝 Vocab örnekleri:")
sample_tokens = list(existing_vocab.items())[:10]
for token, token_id in sample_tokens:
    print(f"   '{token}': {token_id}")
print("   ...")


## 2. Test Veri Setlerini Yükle


## 4. kokler_v08.json'ı Yükle ve Filtreleme Hazırlığı


In [ ]:
# kokler_v08.json dosyasını yükle
print("📚 kokler_v08.json dosyası yükleniyor...")
with open('kokler_v08.json', 'r', encoding='utf-8') as f:
    kokler_dict = json.load(f)

# Kokler sözlüğündeki tokenleri set olarak kaydet (hızlı arama için)
existing_tokens = set(kokler_dict.keys())
print(f"✅ Kokler sözlüğü yüklendi: {len(existing_tokens):,} token")

# Filtreleme fonksiyonlarını tanımla
import string

def is_valid_token(token):
    """Token'ın geçerli olup olmadığını kontrol eder"""
    
    # Boş string kontrolü
    if not token or len(token.strip()) == 0:
        return False
    
    # kokler_v08.json'da bulunan tokenları çıkar
    if token in existing_tokens:
        return False
    
    # Latin alfabesi + Türkçe karakterler + rakamlar + noktalama + boşluk
    turkish_chars = 'ÇĞIİÖŞÜçğıiöşü'
    valid_chars = string.ascii_letters + turkish_chars + string.digits + string.punctuation + string.whitespace
    
    # Token'daki her karakteri kontrol et
    for char in token:
        if char not in valid_chars:
            return False
    
    # Tekrarlanan noktalama işaretlerini filtrele
    if len(token) > 1 and all(c in string.punctuation for c in token):
        # Tüm karakteri aynı noktalama işareti ise (örn: "!!", "...", "---")
        if len(set(token)) == 1:
            return False
        # Sadece noktalama işaretlerinden oluşan çok karakterli tokenlar
        return False
    
    # Çoklu rakam kombinasyonlarını filtrele (00, 000, 01, 02, vb.)
    # Sadece tek rakamları kabul et
    if len(token) > 1 and token.isdigit():
        return False
    
    # Sayı benzeri tokenleri filtrele (1., 2., 3. gibi)
    if len(token) > 1 and token[:-1].isdigit() and token[-1] == '.':
        return False
    
    # Kombinasyon noktalama + rakamları filtrele
    if len(token) > 1:
        has_digit = any(c.isdigit() for c in token)
        has_punct = any(c in string.punctuation for c in token)
        if has_digit and has_punct:
            return False
    
    return True

def clean_token_frequencies(token_frequencies):
    """Token frekanslarını filtreler ve temizler"""
    print("🧹 Token frekansları filtreleniyor...")
    
    original_count = len(token_frequencies)
    filtered_frequencies = {}
    
    removed_categories = {
        'non_latin': 0,
        'existing_in_kokler': 0, 
        'repeated_punctuation': 0,
        'multi_digit': 0,
        'other': 0
    }
    
    turkish_chars = 'ÇĞIİÖŞÜçğıiöşü'
    valid_chars = string.ascii_letters + turkish_chars + string.digits + string.punctuation + string.whitespace
    
    for token, freq in token_frequencies.items():
        if not is_valid_token(token):
            # Detaylı kategorilere ayır
            if token in existing_tokens:
                removed_categories['existing_in_kokler'] += 1
            elif not all(c in valid_chars for c in token):
                removed_categories['non_latin'] += 1
            elif len(token) > 1 and token.isdigit():
                removed_categories['multi_digit'] += 1 
            elif len(token) > 1 and all(c in string.punctuation for c in token):
                removed_categories['repeated_punctuation'] += 1
            else:
                removed_categories['other'] += 1
        else:
            filtered_frequencies[token] = freq
    
    filtered_count = len(filtered_frequencies)
    removed_count = original_count - filtered_count
    
    print(f"📊 Filtreleme sonuçları:")
    print(f"   Orijinal token sayısı: {original_count:,}")
    print(f"   Filtrelenmiş token sayısı: {filtered_count:,}")
    print(f"   Çıkarılan token sayısı: {removed_count:,}")
    print(f"   📋 Çıkarılan tokenların kategorileri:")
    print(f"      - kokler_v08.json'da mevcut: {removed_categories['existing_in_kokler']:,}")
    print(f"      - Latin alfabesi dışı karakterler: {removed_categories['non_latin']:,}")
    print(f"      - Tekrarlanan noktalama: {removed_categories['repeated_punctuation']:,}")
    print(f"      - Çoklu rakam kombinasyonları: {removed_categories['multi_digit']:,}")
    print(f"      - Diğer: {removed_categories['other']:,}")
    
    return filtered_frequencies

print("✅ Filtreleme fonksiyonları hazırlandı")


In [ ]:
# Token frekanslarını filtrele
filtered_token_frequencies = clean_token_frequencies(token_frekanslari)

# Filtrelenmiş tokenleri frekansa göre sırala
sirali_tokenlar = sorted(filtered_token_frequencies.items(), key=lambda x: x[1], reverse=True)

print("🏆 Filtreleme sonrası en sık kullanılan 20 token:")
for i, (token, freq) in enumerate(sirali_tokenlar[:20]):
    print(f"{i+1:2d}. '{token}' - {freq:,} kez")

# Frekans dağılımını analiz et
freqs = list(filtered_token_frequencies.values())
print(f"\n📈 Filtrelenmiş frekans istatistikleri:")
print(f"   En yüksek: {max(freqs):,}")
print(f"   En düşük: {min(freqs):,}")
print(f"   Ortalama: {np.mean(freqs):.1f}")
print(f"   Medyan: {np.median(freqs):.1f}")

# Frekans eşiği belirle (dinamik)
# BPE v10 için 9899 token hedefliyoruz (22869-32767)
BPE_START_ID = 22869
BPE_END_ID = 32767
TARGET_TOKENS = BPE_END_ID - BPE_START_ID + 1

print(f"\n🎯 Hedef token sayısı: {TARGET_TOKENS:,}")

# En sık kullanılan TARGET_TOKENS kadar token al
if len(sirali_tokenlar) > TARGET_TOKENS:
    secili_tokenlar = sirali_tokenlar[:TARGET_TOKENS]
    min_frekans = secili_tokenlar[-1][1]
else:
    secili_tokenlar = sirali_tokenlar
    min_frekans = 1

print(f"📊 Filtrelenmiş token sayısı: {len(sirali_tokenlar):,}")
print(f"📊 Seçilen token sayısı: {len(secili_tokenlar):,}")
print(f"📊 Minimum frekans eşiği: {min_frekans:,}")

if secili_tokenlar:
    frekanslar = [freq for _, freq in secili_tokenlar]
    print(f"\n📊 Seçilen tokenların istatistikleri:")
    print(f"   En yüksek frekans: {max(frekanslar):,}")
    print(f"   En düşük frekans: {min(frekanslar):,}")
    print(f"   Ortalama frekans: {np.mean(frekanslar):.1f}")

print(f"\n✅ Token seçimi tamamlandı!")


In [ ]:
# Frekans analizi için büyük veri setlerini yükle
print("📚 Test veri setleri yükleniyor...")

# Wikipedia veri seti
print("\n🔄 Wikipedia yükleniyor...")
dswiki = load_dataset("wikimedia/wikipedia", "20231101.tr")
dfwiki = dswiki['train'].to_pandas()
print(f"✅ Wikipedia: {len(dfwiki):,} makale")


# Yorumlar veri seti (frekans analizi için)
print("\n🔄 Hepsiburada yorumları yükleniyor...")
dshepsi = load_dataset("alibayram/hepsiburada_yorumlar")
dfhepsi = dshepsi['train'].to_pandas()
print(f"✅ Hepsiburada: {len(dfhepsi):,} yorum")

print(f"\n📊 Toplam veri boyutu: {len(dfwiki) + len(dfhepsi):,} metin")


## 3. Token Frekanslarını Hesapla


In [ ]:
def sample_and_tokenize(df, text_column, max_samples=50000, dataset_name=""):
    """DataFrame'den örneklem alıp tokenize eder"""
    print(f"🔄 {dataset_name} tokenize ediliyor...")
    
    # Örneklem al
    sample_size = min(len(df), max_samples)
    df_sample = df.head(sample_size)
    
    all_tokens = []
    
    for i in tqdm(range(len(df_sample)), desc=f"{dataset_name} işleniyor"):
        try:
            text = df_sample.iloc[i][text_column]
            if isinstance(text, str) and len(text.strip()) > 0:
                # Çok uzun metinleri böl
                if len(text) > 1000:
                    text = text[:1000]
                
                # Tokenize et
                encoded = tokenizer.encode(text)
                tokens = encoded.tokens
                all_tokens.extend(tokens)
                
        except Exception as e:
            continue
    
    print(f"✅ {dataset_name}: {len(all_tokens):,} token")
    return all_tokens

# Token frekanslarını hesapla
print("📊 Token frekansları hesaplanıyor...\n")
token_frekanslari = Counter()

# Wikipedia tokenları
wiki_tokens = sample_and_tokenize(dfwiki, 'text', max_samples=30000, dataset_name="Wikipedia")
token_frekanslari.update(wiki_tokens)

# Yorum tokenları
yorum_tokens = sample_and_tokenize(dfhepsi, 'Yorum', max_samples=100000, dataset_name="Yorumlar")
token_frekanslari.update(yorum_tokens)

print(f"\n📊 Frekans analizi tamamlandı:")
print(f"   Toplam farklı token: {len(token_frekanslari):,}")
print(f"   Toplam token sayısı: {sum(token_frekanslari.values()):,}")


## 4. En Sık Kullanılan Tokenleri Belirle


In [ ]:
# En sık kullanılan tokenleri sırala
sirali_tokenlar = sorted(token_frekanslari.items(), key=lambda x: x[1], reverse=True)

print("🏆 En sık kullanılan 20 token:")
for i, (token, freq) in enumerate(sirali_tokenlar[:20]):
    print(f"{i+1:2d}. '{token}' - {freq:,} kez")

# Frekans dağılımını analiz et
freqs = list(token_frekanslari.values())
print(f"\n📈 Frekans istatistikleri:")
print(f"   En yüksek: {max(freqs):,}")
print(f"   En düşük: {min(freqs):,}")
print(f"   Ortalama: {np.mean(freqs):.1f}")
print(f"   Medyan: {np.median(freqs):.1f}")

# Frekans eşiği belirle (dinamik)
# BPE v10 için 9899 token hedefliyoruz (22869-32767)
BPE_START_ID = 22869
BPE_END_ID = 32767
TARGET_TOKENS = BPE_END_ID - BPE_START_ID + 1

print(f"\n🎯 Hedef token sayısı: {TARGET_TOKENS:,}")

# En sık kullanılan TARGET_TOKENS kadar token al
if len(sirali_tokenlar) > TARGET_TOKENS:
    secili_tokenlar = sirali_tokenlar[:TARGET_TOKENS]
    min_frekans = secili_tokenlar[-1][1]
else:
    secili_tokenlar = sirali_tokenlar
    min_frekans = 1

print(f"📊 Seçilen token sayısı: {len(secili_tokenlar):,}")
print(f"📊 Minimum frekans eşiği: {min_frekans:,}")

if secili_tokenlar:
    frekanslar = [freq for _, freq in secili_tokenlar]
    print(f"\n📊 Seçilen tokenların istatistikleri:")
    print(f"   En yüksek frekans: {max(frekanslar):,}")
    print(f"   En düşük frekans: {min(frekanslar):,}")
    print(f"   Ortalama frekans: {np.mean(frekanslar):.1f}")


## 5. BPE v10 JSON Dosyasını Oluştur


In [ ]:
# BPE v10 dictionary oluştur
print("📝 BPE v10 dictionary oluşturuluyor...")

bpe_v10_dict = {}
current_id = BPE_START_ID

# Seçilen tokenleri ID'leriyle birlikte sözlüğe ekle
for i, (token, freq) in enumerate(secili_tokenlar):
    if current_id <= BPE_END_ID:
        bpe_v10_dict[token] = current_id
        current_id += 1
    else:
        break

print(f"✅ BPE v10 dictionary oluşturuldu")
print(f"📊 Token sayısı: {len(bpe_v10_dict):,}")
print(f"🔢 Kullanılan ID aralığı: {BPE_START_ID} - {current_id - 1}")

# JSON dosyasını kaydet
output_file = 'bpe_v10.json'
print(f"\n💾 {output_file} dosyasına kaydediliyor...")

with open(output_file, 'w', encoding='utf-8') as f:
    json.dump(bpe_v10_dict, f, ensure_ascii=False, indent=4, sort_keys=True)

print(f"✅ {output_file} başarıyla oluşturuldu!")
print(f"📊 Dosya boyutu: {os.path.getsize(output_file) / 1024:.1f} KB")

# İlk 10 tokenı göster
print(f"\n📝 İlk 10 BPE v10 tokeni:")
first_10 = list(bpe_v10_dict.items())[:10]
for token, token_id in first_10:
    print(f"   '{token}': {token_id}")
print("   ...")


## 6. Sonuç Analizi ve Kalite Kontrolü


In [ ]:
# İstatistikleri hesapla
token_lengths = [len(token) for token in bpe_v10_dict.keys()]
avg_token_length = np.mean(token_lengths)
max_token_length = max(token_lengths)
min_token_length = min(token_lengths)

print("📈 BPE v10 İstatistikleri:")
print(f"   Toplam token sayısı: {len(bpe_v10_dict):,}")
print(f"   ID aralığı: {BPE_START_ID} - {max(bpe_v10_dict.values())}")
print(f"   Ortalama token uzunluğu: {avg_token_length:.2f} karakter")
print(f"   En uzun token: {max_token_length} karakter")
print(f"   En kısa token: {min_token_length} karakter")

# En uzun ve en kısa tokenleri göster
longest_tokens = [token for token in bpe_v10_dict.keys() if len(token) == max_token_length][:5]
shortest_tokens = [token for token in bpe_v10_dict.keys() if len(token) == min_token_length][:5]

print(f"\n📏 En uzun tokenler ({max_token_length} karakter):")
for token in longest_tokens:
    print(f"   '{token}'")

print(f"\n📏 En kısa tokenler ({min_token_length} karakter):")
for token in shortest_tokens:
    print(f"   '{token}'")

# En sık kullanılan tokenları göster (frekans bilgisiyle)
print(f"\n🏆 En sık kullanılan 15 token (frekansla):")
for i, (token, freq) in enumerate(secili_tokenlar[:15]):
    token_id = bpe_v10_dict[token]
    print(f"{i+1:2d}. ID {token_id}: '{token}' (frekans: {freq:,})")

print(f"\n🎉 BPE v10 oluşturma işlemi tamamlandı!")
print(f"📁 Dosya konumu: {os.path.abspath(output_file)}")
print(f"📊 Frekans analizine dayalı {len(bpe_v10_dict):,} token içeriyor")


In [ ]:
# Test tokenizasyon
test_texts = [
    "Bu bir test metnidir.",
    "Merhaba dünya! Nasılsın?",
    "Türkiye'de yaşıyorum.",
    "Çok güzel bir gün bugün."
]

print("🧪 Test Tokenization:")
for text in test_texts:
    encoded = tokenizer.encode(text)
    tokens = encoded.tokens
    print(f"\n📝 Metin: '{text}'")
    print(f"🔤 Tokenler: {tokens}")
    print(f"🔢 Token sayısı: {len(tokens)}")
    
    # BPE v10'da hangi tokenler var?
    bpe_v10_tokens = [token for token in tokens if token in bpe_v10_dict]
    print(f"✨ BPE v10'da bulunan: {len(bpe_v10_tokens)}/{len(tokens)} token")

print("\n✨ Notebook tamamlandı!")
print("📊 Frekans analizine dayalı BPE v10 dosyası başarıyla oluşturuldu.")
